# 02D - Cross-dataset validation and modeling handoff

Verify the completed A/B/C outputs, preserve distinct datasets and labels, and assemble their verified files at the paths used by modeling. This does not join records or establish disaster/shipment linkage. No dataset is reprocessed here.


In [ ]:
EXPERIMENT_ID = "colab-cpu-v3-clean"
DRIVE_FOLDER = "ChainShield-Colab"
ENABLE_DRIVE = False
BOOTSTRAP_EXPERIMENT = False
ALLOW_PACKAGE_INSTALL = False
ALLOW_DATA_DOWNLOADS = False
ALLOW_PRETRAINED_DOWNLOADS = False
RUN_STAGE = False
SYNC_INTERVAL_SECONDS = 60
VERIFY_PERSISTENCE_FULL = False
VERIFY_CHECKPOINT_PAIRS = False
CLEAR_STALE_WRITER_LOCK = False
CONFIRM_NO_OTHER_WRITER = False
DOWNLOAD_REPORT_ZIP = False

# Isolated plotting workers must not inherit the Colab inline backend.
if RUN_STAGE:
    import os
    os.environ["MPLBACKEND"] = "Agg"


In [ ]:
# This small cell registers the visible stage code; it writes no files.
REGISTERED_FILES = {}
def register_stage_file(line, body):
    name=line.strip()
    if not name or name.startswith("/") or "\\" in name or ":" in name or ".." in name.split("/"):
        raise ValueError("Use a safe relative module name")
    REGISTERED_FILES[name]=body.rstrip("\n")+"\n"
get_ipython().register_magic_function(register_stage_file,"cell","chainshield_file")


### scripts/cross_dataset_handoff.py


In [ ]:
%%chainshield_file scripts/cross_dataset_handoff.py
from pathlib import Path
import json,shutil,sys
ROOT=Path(__file__).resolve().parents[1]
sys.path.insert(0,str(ROOT/"src"))
from chainshield.research.preparation import sha
DATASETS=("dataco","india_nepal","crisismmd")

def verify_stage(path):
    marker=json.loads((path/"completion.json").read_text())
    if sha(path/"preparation-manifest.json")!=marker["manifest_sha256"]:raise ValueError("Preparation manifest changed")
    record=json.loads((path/"preparation-manifest.json").read_text())
    for name,expected in record["file_sha256"].items():
        source=(path/name).resolve();source.relative_to(path.resolve())
        if sha(source)!=expected:raise ValueError("Prepared file changed")
    return record

def assemble():
    stages=ROOT/"data/processed/research-v2/dataset_stages"
    records={}
    for dataset in DATASETS:
        records[dataset]=verify_stage(stages/dataset)
        for variant in (("control","url_mentions") if dataset=="crisismmd" else ("control",)):
            folder=ROOT/"artifacts/research-v2/eda"/(dataset+"_"+variant)
            for marker_name in ("completion.json","visualization-completion-v2.json"):
                record=json.loads((folder/marker_name).read_text())
                for name,digest in record["file_sha256"].items():
                    if sha(folder/name)!=digest:raise ValueError("Dataset EDA changed")
    output=ROOT/"data/processed/research-v2/paired_v2"
    if (output/"completion.json").exists():verify_stage(output);return
    if output.exists():raise FileExistsError("Partial aggregate: restore clean temporary runtime")
    output.mkdir(parents=True)
    for variant in ("control","url_mentions"):
        for dataset in DATASETS:shutil.copytree(stages/dataset/variant/dataset,output/variant/dataset)
    record={k:v for k,v in records["dataco"].items() if k not in ("file_sha256","reference","sources","cohort_summaries")}
    record.update(reference={d:records[d]["reference"][d] for d in DATASETS},
                  sources={d:records[d]["sources"][d] for d in DATASETS},
                  cohort_summaries={v:{d:records[d]["cohort_summaries"][v][d] for d in DATASETS} for v in ("control","url_mentions")},
                  file_sha256={p.relative_to(output).as_posix():sha(p) for p in output.rglob("*") if p.is_file()},
                  assembly_policy="Byte-copy independently validated dataset outputs; no reprocessing or split changes")
    (output/"preparation-manifest.json").write_text(json.dumps(record,indent=2)+"\n")
    report=ROOT/"docs/dataset-readiness.md"
    report.write_text("\n\n".join((ROOT/"docs"/("dataset-readiness-"+d+".md")).read_text() for d in DATASETS))
    (output/"completion.json").write_text(json.dumps({"manifest_sha256":sha(output/"preparation-manifest.json"),"model_training":False,"test_scoring":False})+"\n")
    verify_stage(output);print("All three datasets and EDA verified; modeling inputs assembled.",flush=True)

if __name__=="__main__":assemble()


In [ ]:
import hashlib,json
EXPECTED_STAGE_FILES={'scripts/cross_dataset_handoff.py': '3af985763e4481531db59d8b6f9fb6c5b12b552a6630eab59c719637697edaa7'}
assert set(REGISTERED_FILES)==set(EXPECTED_STAGE_FILES)
for name,expected in EXPECTED_STAGE_FILES.items():
    if hashlib.sha256(REGISTERED_FILES[name].encode()).hexdigest()!=expected:raise ValueError("Changed source: "+name)


In [ ]:
# Connect the Colab experiment and load the shared runtime.
from pathlib import Path
CONTROLLER_SHA = '8c092c59baffc0804aeca86b4f0ac3fb6d4be3cd96eea96c9e7299e559d9c9e6'
IMPLEMENTATION_SHA256 = CONTROLLER_SHA
rt = None
if BOOTSTRAP_EXPERIMENT:
    if not ENABLE_DRIVE:
        raise PermissionError("Set ENABLE_DRIVE=True")
    from google.colab import drive
    drive.mount("/content/drive")
    runtime_path=Path("/content/drive/MyDrive")/DRIVE_FOLDER/"experiments"/EXPERIMENT_ID/"workspace/.replication/controller.py"
    if not runtime_path.is_file():
        raise RuntimeError("Complete Version 3 notebook 00 first; its shared controller is missing")
    runtime_code=runtime_path.read_text(encoding="utf-8")
    if hashlib.sha256(runtime_code.encode()).hexdigest()!=CONTROLLER_SHA:
        raise ValueError("Shared runtime version differs; do not mix notebook packages")
    exec(compile(runtime_code,"<Colab shared runtime>","exec"),globals())
    rt=Replica(EXPERIMENT_ID,drive_base=DRIVE_FOLDER,allow_drive=True)
    if CLEAR_STALE_WRITER_LOCK:
        rt.clear_stale_lock(acknowledge_no_other_writer=CONFIRM_NO_OTHER_WRITER)
else:
    print("Bootstrap disabled: no Drive access or filesystem changes")


In [ ]:
if RUN_STAGE:
    if rt is None:raise RuntimeError("Enable Drive/bootstrap")
    for stage in ("02A","02B","02C"):rt.require_stage(stage)
    rt.install_sources(REGISTERED_FILES)
    rt.environment(allow_install=ALLOW_PACKAGE_INSTALL)
    rt.execute("02D-handoff",["scripts/cross_dataset_handoff.py"],SYNC_INTERVAL_SECONDS)
    rt.finish_stage("02D")
